In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
logging.basicConfig(level=logging.ERROR, format='%(levelname)s: %(message)s')

import os
import sys

import numpy as np
import tensorflow as tf # type: ignore

from meridian.model import model
from meridian.model import spec
from meridian.analysis import optimizer
from meridian.analysis import analyzer

from meridian.planner.flex_budget_planner import FlexibleBudgetPlanner

In [3]:
# Optimizer input excel file path
home_dir = '/Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer'
# input_file_path = f'{home_dir}/input_files/optimizer_input_case_coeff.xlsx'
input_file_path = f'{home_dir}/input_files/lmmm_simulated_nat_data_roi.xlsx'

if not os.path.exists(input_file_path):
  raise FileNotFoundError(f'File not found: {input_file_path}')

In [5]:
# Configuration for input excel file
# input_config = {

#   # time and geo inputs
#   'time_col': 'week',
#   'geo_col': 'geo',
#   'population_col': 'population',

#   # kpi inputs
#   'kpi_col': 'conversions',  #
#   'kpi_type': 'non_revenue',
#   'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

#   # impression based media inputs
#   'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
#   'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
#   'media_channels': ['Channel0', 'Channel1', 'Channel2'],

#   # reach based media inputs
#   'reach_cols': ['Channel3_reach'],
#   'frequency_cols': ['Channel3_frequency'],
#   'rf_spend_cols': ['Channel3_spend'],
#   'rf_channels': ['Channel3'],


#   }

input_config = {

  # time and geo inputs
  'time_col': 'week',
  'geo_col': 'geo',
  'population_col': 'population',

  # kpi inputs
  'kpi_col': 'conversions',  #
  'kpi_type': 'non_revenue',
  'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],


  }


In [6]:
optimization_config = {
  'fixed_budget': True,

  # spend constraints
  'spend_constraint_lower': {  # (1 - value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  # 'Channel3': 0.3
  },

  'spend_constraint_upper': {  # (1 + value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  # 'Channel3': 0.3
  },

  # # optimization period
  # 'start_date': '2023-01-02',
  # 'end_date': '2024-01-01'
  'start_date': '2025-01-25',
  'end_date': '2025-03-29'

}

planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
opt_attrs = optimized_data.attrs
optimized_data.to_dataframe().reset_index()

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
I0000 00:00:1758210392.864724 4836017 service.cc:148] XLA service 0x14714eef0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758210392.864758 4836017 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1758210392.874954 4836017 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` a

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-09-18 10:46:33.705918: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,14,0.233333,50.120262,0.358002,3.580019,2.974047,0.279328,mean
1,Channel1,20,0.333333,74.707237,0.373536,3.735362,3.953629,0.267712,mean
2,Channel2,26,0.433333,191.697144,0.737297,7.372967,12.846845,0.135631,mean


In [11]:
opt_attrs

{'start_date': '2025-01-25',
 'end_date': '2025-03-29',
 'budget': np.float32(60.0),
 'profit': np.float32(256.52466),
 'total_incremental_outcome': np.float32(316.52466),
 'total_roi': np.float32(5.275411),
 'total_cpik': np.float32(0.18955868),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': True}

In [19]:
tf.reduce_sum(planner.model_obj.kpi[:, -10:])

<tf.Tensor: shape=(), dtype=float32, numpy=829.6036376953125>

In [20]:
316.52466 / 829.6036376953125

0.3815372132158485

1. Process DataSheet

In [5]:
planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
data = planner.build_input_data()

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
I0000 00:00:1758210144.494777 4831083 service.cc:148] XLA service 0x110921e50 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1758210144.494812 4831083 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1758210144.502930 4831083 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` a

In [10]:
data.reach.shape if data.reach else None

In [11]:
data.frequency.shape if data.frequency else None

In [12]:
print(data.kpi.shape)  # (n_geos, n_times)
print(data.media.shape)  # (n_geos, n_times, n_media_channels)
print(data.geo.shape)  # (n_geos,)
print(data.time.shape)  # (n_times,)
print(data.population.shape)  # (n_geos,)

(1, 117)
(1, 117, 3)
(1,)
(117,)
(1,)


2. Process ParameterSheet

In [13]:
parameter_arrays = planner.get_processed_parameter_arrays()
for param_name, param_array in parameter_arrays.items():
  print(f"{param_name=}", "\n")
  print(param_array)
  print("\n")


param_name='alpha_m' 

<xarray.DataArray 'alpha_m' (media_channel: 3)> Size: 24B
array([0.98, 0.45, 0.68])
Coordinates:
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'


param_name='ec_m' 

<xarray.DataArray 'ec_m' (media_channel: 3)> Size: 24B
array([0.53, 0.88, 1.1 ])
Coordinates:
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'


param_name='slope_m' 

<xarray.DataArray 'slope_m' (media_channel: 3)> Size: 24B
array([ 2.47,  2.55, 11.09])
Coordinates:
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'




3. Process CoefficientsSheet

In [14]:
coefficient_arrays = planner.get_processed_coefficients_arrays()
coefficient_arrays['beta_gm']

<xarray.DataArray 'beta_gm' (geo: 1, media_channel: 3)> Size: 12B
array([[2.786987 , 4.7182198, 8.437376 ]], dtype=float32)
Coordinates:
  * geo            (geo) <U12 48B 'national_geo'
  * media_channel  (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'

In [15]:
coefficient_arrays['beta_gm'].shape

(1, 3)

In [16]:
coefficient_arrays['beta_grf'].shape if 'beta_grf' in coefficient_arrays else None

4. Create a PointInference Object

In [25]:
self = planner
parameter_arrays = self.get_processed_parameter_arrays()
coefficient_arrays = self.get_processed_coefficients_arrays()
input_data_obj = self.build_input_data()

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(


In [27]:
from meridian.planner import point_inference_data
point_data = point_inference_data.PointInferenceData(
        parameter_arrays, coefficient_arrays,
        input_data_obj=input_data_obj
      )

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


In [29]:
point_data.posterior

<xarray.Dataset> Size: 2kB
Dimensions:         (chain: 1, draw: 1, media_channel: 3, geo: 1, time: 117,
                     knots: 1)
Coordinates:
  * chain           (chain) int64 8B 0
  * draw            (draw) int64 8B 0
  * media_channel   (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'
  * geo             (geo) <U12 48B 'national_geo'
  * time            (time) int64 936B 0 1 2 3 4 5 6 ... 111 112 113 114 115 116
  * knots           (knots) int64 8B 0
Data variables:
    alpha_m         (chain, draw, media_channel) float32 12B 0.98 0.45 0.68
    ec_m            (chain, draw, media_channel) float32 12B 0.53 0.88 1.1
    slope_m         (chain, draw, media_channel) float32 12B 2.47 2.55 11.09
    beta_gm         (chain, draw, geo, media_channel) float32 12B 2.787 ... 8...
    mu_t            (chain, draw, time) float32 468B 0.0 0.0 0.0 ... 0.0 0.0 0.0
    knot_values     (chain, draw, knots) float32 4B 0.0
    tau_g           (chain, draw, geo) float32 4B 0.0
    roi_m           (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    mroi_m          (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    contribution_m  (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    beta_m          (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    eta_m           (chain, draw, media_channel) float32 12B 0.0 0.0 0.0

In [30]:
inference_data = point_data.get_inference_data()
inference_data

Inference data with groups:
	> posterior
	> sample_stats

In [18]:
point_inference_data = planner.get_inference_data()
point_inference_data


/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/data/input_data_builder.py:722: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


Inference data with groups:
	> posterior
	> sample_stats

5. Scaling InputData & Borrow other utils from Meridian

In [17]:
model_obj = model.Meridian(
    input_data=data,
    model_spec=spec.ModelSpec(knots=1),
    inference_data=point_inference_data
)
model_obj.sample_prior(n_draws=100, seed=42)

In [18]:
model_obj.kpi_transformer.population_scaled_mean

<tf.Tensor: shape=(), dtype=float32, numpy=83.07044982910156>

In [19]:
model_obj.kpi_transformer.population_scaled_stdev

<tf.Tensor: shape=(), dtype=float32, numpy=2.72761607170105>

In [20]:
model_obj.kpi_transformer._population

<tf.Tensor: shape=(2,), dtype=float32, numpy=array([1., 1.], dtype=float32)>

In [21]:
model_obj.kpi_scaled

<tf.Tensor: shape=(2, 117), dtype=float32, numpy=
array([[-0.9060734 ,  0.46627533,  0.03300569, -0.9011113 , -0.5526803 ,
         0.38676238, -0.2438645 , -1.4485806 , -1.231364  ,  0.15669592,
         2.1709857 ,  1.1342238 ,  0.18611014, -0.20842256, -0.88179463,
        -0.46724033,  0.3529539 ,  1.1400726 ,  0.33667484,  0.633575  ,
        -0.9652011 , -1.1212033 , -2.166916  , -0.9266264 , -1.3373655 ,
        -0.5102764 ,  0.42713282, -0.3856743 , -0.82637304,  0.42950755,
         0.57224596,  0.03648807, -1.3894893 ,  1.0525823 ,  1.7958425 ,
         2.4891634 , -0.10680418,  0.32406834,  0.8096632 , -0.3416033 ,
        -0.4761127 , -0.2677461 , -0.5219151 , -1.1787535 , -0.3700861 ,
        -0.62953323, -0.72947335, -1.9166825 , -1.1420753 , -0.79873216,
        -0.7493495 , -0.201841  ,  1.173909  ,  1.0577765 ,  0.6727483 ,
        -0.01989292,  0.6685191 ,  1.5287145 ,  1.6319916 ,  1.2698632 ,
         0.6388783 ,  0.5989023 , -0.20210952, -0.21623483, -1.3019067 ,
 

In [22]:
model_obj.media_tensors.media_transformer._scale_factors_gm

<tf.Tensor: shape=(2, 3), dtype=float32, numpy=
array([[20.135593, 19.933535, 20.265293],
       [20.135593, 19.933535, 20.265293]], dtype=float32)>

In [23]:
model_obj.media_tensors.media_scaled

<tf.Tensor: shape=(2, 117, 3), dtype=float32, numpy=
array([[[0.9661663 , 0.9839545 , 1.0265459 ],
        [0.9207507 , 1.0689793 , 1.0347117 ],
        [0.9439883 , 0.99400026, 0.92758363],
        [0.9743503 , 0.95752794, 0.99221045],
        [1.1098284 , 1.0319173 , 0.9882946 ],
        [0.97872615, 1.1195135 , 0.9980713 ],
        [1.2066426 , 1.        , 0.9040634 ],
        [0.9300858 , 1.0040438 , 0.9594169 ],
        [0.9223113 , 0.91794294, 0.9970648 ],
        [1.0760689 , 1.087007  , 1.0761653 ],
        [0.8986156 , 1.0708874 , 1.0052425 ],
        [1.0198061 , 1.0275176 , 1.0337423 ],
        [0.9456945 , 0.9516925 , 1.0237765 ],
        [0.93098086, 1.0332588 , 0.9595538 ],
        [1.0456626 , 1.017292  , 1.0387762 ],
        [0.9217098 , 0.998143  , 1.0273719 ],
        [0.98610336, 1.1216959 , 1.1079466 ],
        [1.0924193 , 0.95500976, 1.0362443 ],
        [0.9676611 , 1.1028439 , 0.9660532 ],
        [0.9628795 , 1.0166713 , 1.017081  ],
        [0.999517  , 1.0386

In [24]:
model_obj.inference_data.posterior

<xarray.Dataset> Size: 2kB
Dimensions:         (chain: 1, draw: 1, media_channel: 3, geo: 2, time: 117,
                     knots: 1)
Coordinates:
  * chain           (chain) int64 8B 0
  * draw            (draw) int64 8B 0
  * media_channel   (media_channel) <U8 96B 'Channel0' 'Channel1' 'Channel2'
  * geo             (geo) <U4 32B 'geo0' 'geo1'
  * time            (time) int64 936B 0 1 2 3 4 5 6 ... 111 112 113 114 115 116
  * knots           (knots) int64 8B 0
Data variables:
    alpha_m         (chain, draw, media_channel) float32 12B 0.99 0.38 0.69
    ec_m            (chain, draw, media_channel) float32 12B 0.55 1.16 1.11
    slope_m         (chain, draw, media_channel) float32 12B 2.36 2.52 12.46
    beta_gm         (chain, draw, geo, media_channel) float32 24B 3.738 ... 8...
    mu_t            (chain, draw, time) float32 468B 0.0 0.0 0.0 ... 0.0 0.0 0.0
    knot_values     (chain, draw, knots) float32 4B 0.0
    tau_g           (chain, draw, geo) float32 8B 0.0 0.0
    roi_m           (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    mroi_m          (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    contribution_m  (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    beta_m          (chain, draw, media_channel) float32 12B 0.0 0.0 0.0
    eta_m           (chain, draw, media_channel) float32 12B 0.0 0.0 0.0

In [25]:
model_obj.kpi_transformer.population_scaled_mean, model_obj.kpi_transformer.population_scaled_stdev

(<tf.Tensor: shape=(), dtype=float32, numpy=83.07044982910156>,
 <tf.Tensor: shape=(), dtype=float32, numpy=2.72761607170105>)

6. Optimization

a) Scenario 1: Optimize Historical Budget

In [26]:
optimization_config = {
  'fixed_budget': True,

  # spend constraints
  'spend_constraint_lower': {  # (1 - value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  # 'Channel3': 0.3
  },

  'spend_constraint_upper': {  # (1 + value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  # 'Channel3': 0.3
  },

  # # optimization period
  # 'start_date': '2023-01-02',
  # 'end_date': '2024-01-01'
  'start_date': '2025-01-25',
  'end_date': '2025-03-29'

}

planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
opt_attrs = optimized_data.attrs
optimized_data.to_dataframe().reset_index()

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-09-17 15:12:10.903236: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,29,0.237705,137.630737,0.474589,4.745887,3.854686,0.210709,mean
1,Channel1,41,0.336066,174.007751,0.424409,4.244092,6.135113,0.235622,mean
2,Channel2,52,0.426230,384.518280,0.739458,7.394582,13.411420,0.135234,mean


In [27]:
opt_attrs

{'start_date': '2025-01-25',
 'end_date': '2025-03-29',
 'budget': np.float32(122.0),
 'profit': np.float32(574.15674),
 'total_incremental_outcome': np.float32(696.15674),
 'total_roi': np.float32(5.706203),
 'total_cpik': np.float32(0.1752479),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': True}

In [28]:
opt_results.optimization_grid.spend_grid

<xarray.DataArray 'spend_grid' (grid_spend_index: 25, channel: 3)> Size: 600B
array([[29., 29., 28.],
       [30., 30., 29.],
       [31., 31., 30.],
       [32., 32., 31.],
       [33., 33., 32.],
       [34., 34., 33.],
       [35., 35., 34.],
       [36., 36., 35.],
       [37., 37., 36.],
       [38., 38., 37.],
       [39., 39., 38.],
       [40., 40., 39.],
       [41., 41., 40.],
       [42., 42., 41.],
       [43., 43., 42.],
       [44., 44., 43.],
       [45., 45., 44.],
       [46., 46., 45.],
       [47., 47., 46.],
       [48., 48., 47.],
       [49., 49., 48.],
       [50., 50., 49.],
       [51., 51., 50.],
       [52., 52., 51.],
       [53., 53., 52.]])
Coordinates:
  * grid_spend_index  (grid_spend_index) int64 200B 0 1 2 3 4 ... 20 21 22 23 24
  * channel           (channel) object 24B 'Channel0' 'Channel1' 'Channel2'

In [29]:
opt_results.optimization_grid.incremental_outcome_grid

<xarray.DataArray 'incremental_outcome_grid' (grid_spend_index: 25, channel: 3)> Size: 600B
array([[137.6307373 ,  96.59674072,   1.18696594],
       [141.40509033, 103.04357147,   1.83509064],
       [144.95904541, 109.54502106,   2.79328918],
       [148.30335999, 116.08286285,   4.18904877],
       [151.44900513, 122.63978577,   6.19217682],
       [154.40701294, 129.19921875,   9.02423859],
       [157.18807983, 135.74551392,  12.96646881],
       [159.80282593, 142.26422119,  18.36399078],
       [162.26138306, 148.74176025,  25.62220001],
       [164.57345581, 155.16567993,  35.18986511],
       [166.74835205, 161.5246582 ,  47.52278137],
       [168.79486084, 167.80844116,  63.02471161],
       [170.72129822, 174.00775146,  81.96676636],
       [172.53546143, 180.11448669, 104.39807129],
       [174.24468994, 186.1214447 , 130.06994629],
       [175.85583496, 192.0223999 , 158.40396118],
       [177.37532043, 197.81202698, 188.52545166],
       [178.80912781, 203.48596191, 219.36526489],
       [180.16290283, 209.04046631, 249.80783081],
       [181.44180298, 214.47271729, 278.84146118],
       [182.6506958 , 219.78036499, 305.67376709],
       [183.79412842, 224.96200562, 329.78890991],
       [184.8762207 , 230.0165863 , 350.94604492],
       [185.90090942, 234.943573  , 369.13604736],
       [186.87193298, 239.74301147, 384.51828003]])
Coordinates:
  * grid_spend_index  (grid_spend_index) int64 200B 0 1 2 3 4 ... 20 21 22 23 24
  * channel           (channel) object 24B 'Channel0' 'Channel1' 'Channel2'

In [30]:
data.kpi[:, -10:].sum()

<xarray.DataArray 'kpi' ()> Size: 8B
array(1659.426755)

In [31]:
opt_results.nonoptimized_data.sel(metric='mean').to_dataframe().reset_index()

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,41,0.336066,170.721298,0.416393,4.163934,1.846165,0.240157,mean
1,Channel1,41,0.336066,174.007751,0.424409,4.244092,6.135113,0.235622,mean
2,Channel2,40,0.327869,81.966766,0.204917,2.049169,21.401102,0.488003,mean


In [32]:
opt_attrs

{'start_date': '2025-01-25',
 'end_date': '2025-03-29',
 'budget': np.float32(122.0),
 'profit': np.float32(574.15674),
 'total_incremental_outcome': np.float32(696.15674),
 'total_roi': np.float32(5.706203),
 'total_cpik': np.float32(0.1752479),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': True}

In [33]:
opt_results.nonoptimized_data.sel(metric='mean').attrs

{'start_date': '2025-01-25',
 'end_date': '2025-03-29',
 'budget': np.float32(122.0),
 'profit': np.float32(304.69583),
 'total_incremental_outcome': np.float32(426.69583),
 'total_roi': np.float32(3.4975069),
 'total_cpik': np.float32(0.28591797),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True}

In [34]:
display(opt_results.plot_response_curves())
display(opt_results.plot_incremental_outcome_delta())
display(opt_results.plot_spend_delta())
opt_attrs

alt.FacetChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

{'start_date': '2025-01-25',
 'end_date': '2025-03-29',
 'budget': np.float32(122.0),
 'profit': np.float32(574.15674),
 'total_incremental_outcome': np.float32(696.15674),
 'total_roi': np.float32(5.706203),
 'total_cpik': np.float32(0.1752479),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': True}

In [37]:
696.15674 / 1664


0.4183634254807692

b) Scenario 2: Increment Historical Budget By 20%

In [ ]:
optimization_config = {
  'fixed_budget': True,
  'budget': 1.2 * 31310000.0,  # 20% increase in historical budget

  # spend constraints
  'spend_constraint_lower': {  # (1 - value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  },

  'spend_constraint_upper': {  # (1 + value)% of historical
  'Channel0': 0.3,
  'Channel1': 0.3,
  'Channel2': 0.3
  },

  # optimization period
  'start_date': '2023-01-02',
  'end_date': '2024-01-01'

}

planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
opt_attrs = optimized_data.attrs
optimized_data.to_dataframe().reset_index()

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,12290000,0.327123,20708256.0,0.018733,1.684968,1.057250,0.593483,mean
1,Channel1,12310000,0.327655,26938932.0,0.025021,2.188378,1.054687,0.456959,mean
2,Channel2,12970000,0.345222,33598688.0,0.030936,2.590492,1.083608,0.386027,mean


In [55]:
display(opt_results.plot_response_curves())
display(opt_results.plot_incremental_outcome_delta())
display(opt_results.plot_spend_delta())
opt_attrs

alt.FacetChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

{'start_date': '2023-01-02',
 'end_date': '2024-01-01',
 'budget': np.float32(37570000.0),
 'profit': np.float32(43675870.0),
 'total_incremental_outcome': np.float32(81245870.0),
 'total_roi': np.float32(2.16252),
 'total_cpik': np.float32(0.4624235),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': np.False_,
 'fixed_budget': True}

c) Scenario 3: Flex Budget Target ROI

In [ ]:
optimization_config = {
  'fixed_budget': False,
  'target_roi': 1.2,

  # optimization period
  'start_date': '2023-01-02',
  'end_date': '2024-01-01'

}

planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
opt_attrs = optimized_data.attrs
optimized_data.to_dataframe().reset_index()

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,27280000,0.435644,31709088.0,0.012923,1.162357,0.505293,0.860321,mean
1,Channel1,18700000,0.298627,32320182.0,0.019761,1.728352,0.666096,0.578586,mean
2,Channel2,16640000,0.265730,37058868.0,0.026596,2.227095,0.811803,0.449015,mean


In [57]:
optimization_config = {
  'fixed_budget': False,
  'target_roi': 2.0,

  # optimization period
  'start_date': '2023-01-02',
  'end_date': '2024-01-01'

}

planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)
optimized_data = opt_results.optimized_data.sel(metric='mean')
opt_attrs = optimized_data.attrs
optimized_data.to_dataframe().reset_index()

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,14660000,0.335930,23057266.0,0.017486,1.572801,0.921951,0.635808,mean
1,Channel1,14020000,0.321265,28633604.0,0.023351,2.042340,0.921755,0.489634,mean
2,Channel2,14960000,0.342805,35597744.0,0.028417,2.379528,0.921096,0.420251,mean


In [58]:
display(opt_results.plot_response_curves())
display(opt_results.plot_incremental_outcome_delta())
display(opt_results.plot_spend_delta())
opt_attrs

alt.FacetChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

{'start_date': '2023-01-02',
 'end_date': '2024-01-01',
 'budget': np.float32(43640000.0),
 'profit': np.float32(43648616.0),
 'total_incremental_outcome': np.float32(87288616.0),
 'total_roi': np.float32(2.0001974),
 'total_cpik': np.float32(0.49995065),
 'is_revenue_kpi': True,
 'confidence_level': 0.9,
 'use_historical_budget': True,
 'fixed_budget': False,
 'target_roi': 2}

In [59]:
opt_results.spend_bounds

(array([0.]), array([2.]))

In [62]:
nonopt_df = opt_results.nonoptimized_data.sel(metric='mean').to_dataframe().reset_index()
opt_df = opt_results.optimized_data.sel(metric='mean').to_dataframe().reset_index()


In [63]:
nonopt_df

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,13640000,0.435644,22085456.0,0.018002,1.619168,0.976730,0.617601,mean
1,Channel1,9350000,0.298627,23369356.0,0.028577,2.499396,1.366845,0.400097,mean
2,Channel2,8320000,0.265730,27261440.0,0.039130,3.276615,1.701346,0.305193,mean


In [64]:
opt_df

,channel,spend,pct_of_spend,incremental_outcome,effectiveness,roi,mroi,cpik,metric
0,Channel0,14660000,0.335930,23057266.0,0.017486,1.572801,0.921951,0.635808,mean
1,Channel1,14020000,0.321265,28633604.0,0.023351,2.042340,0.921755,0.489634,mean
2,Channel2,14960000,0.342805,35597744.0,0.028417,2.379528,0.921096,0.420251,mean


In [65]:
opt_df['spend'] / nonopt_df['spend']

0    1.074780
1    1.499465
2    1.798077
Name: spend, dtype: float64

In [66]:
opt_df['incremental_outcome'] / nonopt_df['incremental_outcome']

0    1.044002
1    1.225263
2    1.305791
Name: incremental_outcome, dtype: float32